In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import transformers
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")


In [ ]:
CONFIG = {
    'model_name': 'facebook/wav2vec2-base',
    'sample_rate': 16000,
    'max_duration': 6.0,
    'batch_size': 8,
    'epochs': 5,
    'lr': 3e-5,
    'n_splits': 5,
    'seed': 42
}
possible_dirs = [
    Path('/kaggle/input/dataverse-detecting-emotions-from-hindi-speech/SRCASW-BhavVaani'),
    Path('/kaggle/input/dataverse-detecting-emotions-from-a-hindi-speech/SRCASW-BhavVaani'),
    Path('./SRCASW-BhavVaani'),
    Path('../SRCASW-BhavVaani')
]
DATA_DIR = None
for p in possible_dirs:
    if p.exists():
        DATA_DIR = p
        break
if DATA_DIR is None:
    DATA_DIR = Path('.')
print(f"Selected Data Directory: {DATA_DIR}")
train_csv_path = DATA_DIR / 'train.csv' if (DATA_DIR / 'train.csv').exists() else Path('train.csv')
test_csv_path = DATA_DIR / 'test.csv' if (DATA_DIR / 'test.csv').exists() else Path('test.csv')
sample_sub_path = DATA_DIR / 'sample_submission.csv' if (DATA_DIR / 'sample_submission.csv').exists() else Path('sample_submission.csv')
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)
sample_sub = pd.read_csv(sample_sub_path)
print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print("\nTrain Preview:")
display(train_df.head())


In [ ]:
print("Train Class Distribution:")
print(train_df['emotion'].value_counts())
plt.figure(figsize=(7, 4))
train_df['emotion'].value_counts().plot(kind='bar', color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], edgecolor='black')
plt.title("Emotion Class Distribution in Training Set")
plt.xlabel("Emotion")
plt.ylabel("Number of Samples")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
unique_emotions = sorted(train_df['emotion'].unique().tolist())
label2id = {label: i for i, label in enumerate(unique_emotions)}
id2label = {i: label for i, label in enumerate(unique_emotions)}
train_df['label'] = train_df['emotion'].map(label2id)
print(f"Label Mapping: {label2id}")


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(CONFIG['model_name'])
class HindiSpeechDataset(Dataset):
    def __init__(self, df, audio_dir, feature_extractor, max_duration=6.0, is_train=True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.feature_extractor = feature_extractor
        self.target_sr = CONFIG['sample_rate']
        self.max_samples = int(max_duration * self.target_sr)
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = self.audio_dir / row['filename']
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)
        wav_np = waveform.squeeze(0).numpy()
        if len(wav_np) > self.max_samples:
            wav_np = wav_np[:self.max_samples]
        else:
            wav_np = np.pad(wav_np, (0, self.max_samples - len(wav_np)), mode='constant')
        inputs = self.feature_extractor(
            wav_np,
            sampling_rate=self.target_sr,
            return_tensors="pt"
        )
        input_values = inputs.input_values.squeeze(0)
        item = {'input_values': input_values}
        if 'label' in row:
            item['label'] = torch.tensor(row['label'], dtype=torch.long)
        return item
train_audio_dir = DATA_DIR / 'train' if (DATA_DIR / 'train').exists() else Path('train')
test_audio_dir = DATA_DIR / 'test' if (DATA_DIR / 'test').exists() else Path('test')
print(f"Train audio dir: {train_audio_dir} (files found: {len(list(train_audio_dir.glob('*.wav')))})")
print(f"Test audio dir: {test_audio_dir} (files found: {len(list(test_audio_dir.glob('*.wav')))})")


In [ ]:
def compute_macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')
skf = StratifiedKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
train_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_df.loc[val_idx, 'fold'] = fold
print("K-Fold Distribution:")
print(train_df.groupby(['fold', 'emotion']).size().unstack())


In [ ]:
def train_one_fold(fold_num, train_subset, val_subset):
    print(f"\n{'='*20} Training Fold {fold_num + 1}/{CONFIG['n_splits']} {'='*20}")
    train_dataset = HindiSpeechDataset(train_subset, train_audio_dir, feature_extractor, max_duration=CONFIG['max_duration'])
    val_dataset = HindiSpeechDataset(val_subset, train_audio_dir, feature_extractor, max_duration=CONFIG['max_duration'], is_train=False)
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    model = AutoModelForAudioClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=len(unique_emotions),
        label2id=label2id,
        id2label=id2label
    )
    if hasattr(model, 'freeze_feature_encoder'):
        model.freeze_feature_encoder()
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_macro_f1 = 0.0
    best_val_preds = []
    best_model_path = f"best_model_fold_{fold_num}.pt"
    for epoch in range(CONFIG['epochs']):
        model.train()
        running_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Fold {fold_num+1} Epoch {epoch+1}/{CONFIG['epochs']} [Train]", leave=False):
            optimizer.zero_grad()
            input_values = batch['input_values'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_values=input_values)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(labels)
        train_loss = running_loss / len(train_dataset)
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_values = batch['input_values'].to(device)
                labels = batch['label'].to(device)
                outputs = model(input_values=input_values)
                preds = torch.argmax(outputs.logits, dim=-1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
        val_macro_f1 = compute_macro_f1(val_targets, val_preds)
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_val_preds = val_preds
            torch.save(model.state_dict(), best_model_path)
    print(f"Fold {fold_num + 1} Best Macro F1: {best_macro_f1:.4f}")
    return best_model_path, best_val_preds
FOLDS_TO_RUN = 5
trained_models = []
oof_predictions = np.zeros(len(train_df))
for fold in range(FOLDS_TO_RUN):
    train_sub = train_df[train_df['fold'] != fold]
    val_sub = train_df[train_df['fold'] == fold]
    model_path, best_preds = train_one_fold(fold, train_sub, val_sub)
    trained_models.append(model_path)
    oof_predictions[val_sub.index] = best_preds


In [ ]:
overall_f1 = compute_macro_f1(train_df['label'], oof_predictions)
print(f"\n{'='*20} Overall Cross-Validation Performance {'='*20}")
print(f"Overall OOF Macro F1 Score: {overall_f1:.4f}\n")
print(classification_report(train_df['label'], oof_predictions, target_names=unique_emotions))
cm = confusion_matrix(train_df['label'], oof_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap='Blues')
plt.title('Out-of-Fold Confusion Matrix', y=1.1)
fig.colorbar(cax)
ax.set_xticks(range(len(unique_emotions)))
ax.set_yticks(range(len(unique_emotions)))
ax.set_xticklabels(unique_emotions)
ax.set_yticklabels(unique_emotions)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
for i in range(len(unique_emotions)):
    for j in range(len(unique_emotions)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black' if cm[i,j] < cm.max()/2 else 'white')
plt.show()


In [ ]:
test_dataset = HindiSpeechDataset(test_df, test_audio_dir, feature_extractor, max_duration=CONFIG['max_duration'], is_train=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
test_probs = np.zeros((len(test_df), len(unique_emotions)))
for fold_idx, model_path in enumerate(trained_models):
    print(f"Running inference with Model Fold {fold_idx + 1}...")
    model = AutoModelForAudioClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=len(unique_emotions),
        label2id=label2id,
        id2label=id2label
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    fold_probs = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Predicting Fold {fold_idx+1}", leave=False):
            input_values = batch['input_values'].to(device)
            outputs = model(input_values=input_values)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            fold_probs.append(probs)
    test_probs += np.vstack(fold_probs) / len(trained_models)
final_preds = np.argmax(test_probs, axis=-1)
predicted_emotions = [id2label[p] for p in final_preds]
print("Inference completed.")


In [ ]:
sub_df = pd.DataFrame({
    'Id': test_df['Id'],
    'emotion': predicted_emotions
})
sub_df.to_csv('submission.csv', index=False)
print("Saved submission.csv")
print(f"Rows: {len(sub_df)} (Expected: 209)")
display(sub_df.head(10))
solutions_df = pd.DataFrame({
    'AUDIO_ID': [f"{int(x):04d}" for x in test_df['Id']],
    'PREDICTED LABEL': [e.capitalize() for e in predicted_emotions]
}).sort_values('AUDIO_ID')
solutions_df.to_csv('solutions.csv', index=False)
print("Saved solutions.csv")
display(solutions_df.head(10))
print("\nTest Set Prediction Distribution:")
print(pd.Series(predicted_emotions).value_counts())
